In [1]:
# Load the Matlab Data into Python
# Implement EEGNet for two class classification using GPU.
import os
import glob
import numpy as np
from scipy.io import loadmat
from scipy import signal 
from sklearn.model_selection import LeaveOneOut
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

In [2]:
def load_mat_data(all_files):
    # Load the training data
    X = []
    Y = []
    for filename in all_files:
        data = loadmat(filename)
        X.append(data['X'])
        Y.append(data['Y'])
        fs = data['fs']
    
    # Concatenate training data
    X_data = np.concatenate(X, axis=0)
    Y_data = np.concatenate(Y, axis=0)

    return X_data, Y_data, fs

# Baseline Correction 
def baseline_correction(X, baseline_samples=500):
    n_trails, n_samples, n_channels = X.shape
    Xbc = np.zeros_like(X)
    for t in range(n_trails):
        Xbase = X[t, :baseline_samples-1,:]
        Xeeg = X[t, baseline_samples:, :]
        Xbc[t, baseline_samples:, :] = Xeeg - np.mean(Xbase, axis=0) #baseline correction
    
    Xnew = Xbc[:, baseline_samples:, :]
    return Xnew

# Preprocessing: Surface Laplacian, Bandpass Filter
def bandpass_filtering(X, fs=500, fcut=[0.5, 45], filt_order=5):
    n_trials, n_samples, n_channels = X.shape
    X1 = np.zeros_like(X)
    b,a = signal.butter(filt_order, fcut, fs=fs, btype = 'band', output='ba') 
    for t in range(n_trials):
        for c in range(n_channels):
            #Error here.
            raw_signal = X[t, :, c]
            filt_signal = signal.filtfilt(b, a, raw_signal)
            X1[t, :, c] = filt_signal
            # Xfilt[t, :, c] = signal.filtfilt(b, a, X[t, :, c])
    return X1  



In [3]:
# EEGNet: Handles the convolutional layers
class EEGNet(nn.Module):
    def __init__(self, F1=8, D=2, F2=16, kernLength=64, Chans=27, Samples=2000):
        super(EEGNet, self).__init__()
        
        # Block 1
        self.conv1 = nn.Conv2d(1, F1, (1, kernLength), padding='same', bias=False)
        self.batchnorm1 = nn.BatchNorm2d(F1)
        self.depthwiseConv = nn.Conv2d(F1, F1 * D, (Chans, 1), groups=F1, bias=False)
        self.batchnorm2 = nn.BatchNorm2d(F1 * D)
        self.pool1 = nn.AvgPool2d((1, 4))
        
        # Block 2
        self.separableConv = nn.Conv2d(F1 * D, F2, (1, 16), padding='same', bias=False)
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.pool2 = nn.AvgPool2d((1, 8))

    def forward(self, x):
        # Input shape: (N, Chans, Samples) -> Add channel dimension: (N, 1, Chans, Samples)
        # x = x.unsqueeze(1)

        # Block 1
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = F.elu(x)
        x = self.depthwiseConv(x)
        x = self.batchnorm2(x)
        x = F.elu(x)
        x = self.pool1(x)

        # Block 2
        x = self.separableConv(x)
        x = self.batchnorm3(x)
        x = F.elu(x)
        x = self.pool2(x)

        return x

# SENet: Handles the squeeze-and-excitation
class SENet(nn.Module):
    def __init__(self, F2):
        super(SENet, self).__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)  # Global Average Pooling
        self.fc1 = nn.Linear(F2, F2 // 4, bias=False)  # Squeeze
        self.fc2 = nn.Linear(F2 // 4, F2, bias=False)  # Excitation

    def forward(self, x):
        b, c, _, _ = x.size()  # (batch_size, channels, height, width)
        se = self.global_avg_pool(x).view(b, c)  # Global average pooling and reshape
        se = F.relu(self.fc1(se))  # Squeeze
        se = torch.sigmoid(self.fc2(se))  # Excitation
        se = se.view(b, c, 1, 1)  # Reshape to apply to the original feature map
        x = x * se  # Scale the original feature map by the SE block output
        return x

# Final model: Combines EEGNet, SENet, and Dense layers
class EEGNetWithSE(nn.Module):
    def __init__(self, F1=8, D=2, F2=16, kernLength=64, Chans=27, Samples=2000, 
                 dropoutRate=0.5, norm_rate=1.0, num_classes=2, optional_eegnet=False):
        super(EEGNetWithSE, self).__init__()
        
        self.eegnet = EEGNet(F1, D, F2, kernLength, Chans, Samples)  # Initial EEGNet
        self.senet = SENet(F2)  # SENet
        self.optional_eegnet = optional_eegnet  # Whether to apply EEGNet again
        if optional_eegnet:
            self.eegnet2 = EEGNet(F1, D, F2, kernLength, Chans, Samples)  # Optional second EEGNet

        # Dropout, Dense, and Softmax Layers
        self.dropout = nn.Dropout(dropoutRate)  # Dropout after SE
        self.flatten = nn.Flatten()  # Flatten the output for the fully connected layer
        self.fc = nn.Linear(F2 * (Samples // (4 * 8)), num_classes)  # Fully connected layer
        self.softmax = nn.Softmax(dim=1)  # Softmax activation for classification
        
        # Optional: Apply kernel constraint for the dense layer (weight clipping)
        self.fc.weight = torch.nn.Parameter(
            torch.clamp(self.fc.weight, max=norm_rate)
        )

    def forward(self, x):
        # EEGNet Block
        x = self.eegnet(x)  # Initial EEGNet

        # SENet Block
        x = self.senet(x)  # Squeeze-and-Excitation

        # Optional Second EEGNet Block
        if self.optional_eegnet:
            x = self.eegnet2(x)  # Pass through EEGNet again

        # Dropout and Flatten
        x = self.dropout(x)  # Apply dropout
        x = self.flatten(x)  # Flatten for the fully connected layer

        # Fully connected layer and softmax for classification
        x = self.fc(x)
        x = self.softmax(x)  # Apply softmax activation for 2-class classification
        
        return x

In [4]:
# class EEGNet(nn.Module):
#     def __init__(self, nb_classes, Chans=27, Samples=2500, dropoutRate=0.5, 
#                  kernLength=64, F1=8, D=2, F2=16, norm_rate=0.25, dropoutType='Dropout'):
#         super(EEGNet, self).__init__()
        
#         # Handle dropout type
#         if dropoutType == 'SpatialDropout2D':
#             self.dropout = nn.Dropout2d(dropoutRate)
#         elif dropoutType == 'Dropout':
#             self.dropout = nn.Dropout(dropoutRate)
#         else:
#             raise ValueError('dropoutType must be one of SpatialDropout2D or Dropout.')

#         # Block 1
#         self.conv1 = nn.Conv2d(1, F1, (1, kernLength), padding='same', bias=False)
#         self.batchnorm1 = nn.BatchNorm2d(F1)
#         self.depthwiseConv = nn.Conv2d(F1, F1*D, (Chans, 1), groups=F1, bias=False)
#         self.batchnorm2 = nn.BatchNorm2d(F1*D)
#         self.pool1 = nn.AvgPool2d((1, 4))

#         # Block 2
#         self.separableConv = nn.Conv2d(F1*D, F2, (1, 16), padding='same', bias=False)
#         self.batchnorm3 = nn.BatchNorm2d(F2)
#         self.pool2 = nn.AvgPool2d((1, 8))

#         # Flatten and Dense
#         self.flatten = nn.Flatten()
#         self.dense = nn.Linear(F2 * (Samples // (4 * 8)), nb_classes)
#         self.norm_constraint = nn.utils.weight_norm(self.dense)

#     def forward(self, x):
#         # Block 1
#         x = self.conv1(x)
#         x = self.batchnorm1(x)
#         x = self.depthwiseConv(x)
#         x = self.batchnorm2(x)
#         x = F.elu(x)
#         x = self.pool1(x)
#         x = self.dropout(x)

#         # Block 2
#         x = self.separableConv(x)
#         x = self.batchnorm3(x)
#         x = F.elu(x)
#         x = self.pool2(x)
#         x = self.dropout(x)

#         # Flatten and Dense
#         x = self.flatten(x)
#         x = self.dense(x)
#         return F.softmax(x, dim=1)

In [5]:
# Function to compute accuracy
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets.squeeze()).sum().item()
            total += targets.size(0)
    
    accuracy = correct / total
    return accuracy


In [6]:
torch.manual_seed(0)
parent_dir = os.path.dirname(os.getcwd())
rel_path = 'data/*midata.mat'
A = glob.glob(os.path.join(parent_dir, rel_path))

batch_size = 32  # Adjust batch size as needed
num_epochs = 500

# Initialize the Leave-One-Out Cross-Validation
loo = LeaveOneOut()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Perform Leave-One-Out Cross-Validation
perf = dict()
for train_index, test_index in loo.split(A):
    # Split data into training and test sets
    train_sub_indx = [A[i] for i in train_index]
    test_sub_indx = [A[i] for i in test_index]

    X_train, Y_train, fs = load_mat_data(train_sub_indx)
    X_train = baseline_correction(X_train)
    X_train = bandpass_filtering(X_train)
    # print(f'Test file: {train_sub_indx[0]}')
    # print(f'X_train shape: {X_train.shape}')
    # print(f'Y_train shape: {Y_train.shape}')


    # print(f'Fold: {test_index[0]}: Test file: {test_sub_indx[0]}')
    X_test, Y_test, fs = load_mat_data(test_sub_indx)
    X_test = baseline_correction(X_test)
    X_test = bandpass_filtering(X_test)
    # print(f'Test file: {test_sub_indx[0]}')
    # print(f'X_test shape: {X_test.shape}')
    # print(f'Y_test shape: {Y_test.shape}')

    # Convert numpy arrays to PyTorch tensors and move to GPU
    # Data loading (assuming X_train, X_test, Y_train, Y_test are already available as NumPy arrays)
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_train_tensor = torch.tensor(Y_train, dtype=torch.long).to(device)  # Use long for classification
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_test_tensor = torch.tensor(Y_test, dtype=torch.long).to(device)

    # print(X_train_tensor.shape)
    
    # X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
    # Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32).to(device)
    # X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
    # Y_test_tensor = torch.tensor(Y_test, dtype=torch.float32).to(device)

    train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)


    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

    # # Example usage
    # nb_classes = 2
    # Chans = 27
    # Samples = 2000
    # dropoutRate = 0.5
    # kernLength = 64
    # F1 = 8
    # D = 2
    # F2 = 16
    # norm_rate = 0.25
    # dropoutType = 'Dropout'

    # model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    model = EEGNetWithSE().to(device)
    # model = EEGNetWithSE(nb_classes=2, Chans=27, Samples=2000).to(device)


    # model = EEGNet().to(device)

    criterion = nn.CrossEntropyLoss().to(device)  # Move loss function to GPU if needed
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            # outputs = model(inputs.permute(0, 2, 1))  # Permute to match Conv1D input shape
            loss = criterion(outputs, targets.squeeze())
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
    
        # print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}')

    # Compute accuracy on the test dataset
    accuracy = compute_accuracy(model, test_loader, device)
    perf[test_index[0]] = accuracy
    
    print(f'Test Subject: {test_sub_indx[0]}')
    print(f'Test Accuracy: {accuracy:.4f}')

print(perf)

d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\modules\conv.py:454: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\Convolution.cpp:1032.)
  return F.conv2d(input, weight, bias, self.stride,


Test Subject: d:\Praveen\PostDoc@SIT\SIT2024\MIDecoding_SENet\data\Ajul_midata.mat
Test Accuracy: 0.4444
Test Subject: d:\Praveen\PostDoc@SIT\SIT2024\MIDecoding_SENet\data\Ananthu_midata.mat
Test Accuracy: 0.5556
Test Subject: d:\Praveen\PostDoc@SIT\SIT2024\MIDecoding_SENet\data\Asish_midata.mat
Test Accuracy: 0.6250
Test Subject: d:\Praveen\PostDoc@SIT\SIT2024\MIDecoding_SENet\data\Aswathy_midata.mat
Test Accuracy: 0.4583
Test Subject: d:\Praveen\PostDoc@SIT\SIT2024\MIDecoding_SENet\data\Athira_midata.mat
Test Accuracy: 0.5000
Test Subject: d:\Praveen\PostDoc@SIT\SIT2024\MIDecoding_SENet\data\Bharath2_midata.mat
Test Accuracy: 0.4028
Test Subject: d:\Praveen\PostDoc@SIT\SIT2024\MIDecoding_SENet\data\Danish_midata.mat
Test Accuracy: 0.5139
Test Subject: d:\Praveen\PostDoc@SIT\SIT2024\MIDecoding_SENet\data\Durga_midata.mat
Test Accuracy: 0.5000
Test Subject: d:\Praveen\PostDoc@SIT\SIT2024\MIDecoding_SENet\data\Gayathri_midata.mat
Test Accuracy: 0.5000
Test Subject: d:\Praveen\PostDoc@SI

In [8]:
# Without Preprocessing
print(perf.values())


dict_values([0.5277777777777778, 0.5, 0.5555555555555556, 0.5833333333333334, 0.4861111111111111, 0.5277777777777778, 0.5833333333333334, 0.4722222222222222, 0.5277777777777778, 0.4861111111111111, 0.5972222222222222, 0.4722222222222222, 0.5555555555555556, 0.5416666666666666, 0.4861111111111111, 0.4722222222222222, 0.5, 0.5, 0.5555555555555556, 0.5, 0.4444444444444444])


In [ ]:
print(perf.values())


In [7]:
mean_acc = list(perf.values())
np.mean(mean_acc)

0.5462962962962964

Results Summarization

Methods:
    x = self.eegnet(x)  # Initial EEGNet
    x = self.senet(x)  # Squeeze-and-Excitation
    x = self.dropout(x)  # Apply dropout
    x = self.flatten(x)  # Flatten for the fully connected layer
    x = self.fc(x)
    x = self.softmax(x)  # Apply softmax activation for 2-class classification

Parameters:
    1. Batch Size 64, lr=1e-4 and epochs 100
       - Average LOOV Accuracy: 55.82%
    2. Batch Size 32, lr=1e-3 and epoch 500
       - Average LOOV Accuracy: 54.62%